# Is `AngularTwoPoint.get_Cl` autodifferentiable?

`cloelib` computes almost everything in `jax.numpy`, but most of its cosmology backends (`CAMBBackground`, `HMcode2020Emu`, ...) call out to non-JAX external codes internally, which breaks autodiff even where the rest of the pipeline is pure JAX. The one backend that's JAX end-to-end is `cloelib.cosmology.jax_cosmology`: `JAXBackground` (analytic background), `JAXLinearPerturbations` (Eisenstein-Hu transfer function + a JAX ODE solve for the growth factor), `JAXNonLinearPerturbations` (halofit via `jax.vmap`). This notebook uses that backend specifically to ask: is `get_Cl` differentiable via `jax.grad`, for both the default NLA intrinsic-alignment model and TATT?

**Short answer, found empirically below:** yes - `get_Cl()` itself, the public API, is now fully autodifferentiable for NLA, including through cosmological parameters like $H_0$. That wasn't always true: an earlier run of this notebook found that `get_Cl`'s packaging step broke `jax.grad` via an external-package detail unrelated to the physics (`cosmolib.data.photo.AngularPowerSpectrum.__post_init__` forcing a plain NumPy cast) - that's now fixed upstream in `cosmolib` (see Landmine #2 below), so there's no longer a reason to reach for a private, packaging-free method to get a gradient. TATT is differentiable with respect to its own amplitude parameters, but not through the one-loop kernels themselves, because those come from FAST-PT (plain NumPy/SciPy), not JAX - that boundary is structural, not something this notebook's fix touches.


## Setup

No external data needed - this notebook uses a synthetic Gaussian $n(z)$ rather than the FITS files the other tutorials load, to keep the differentiability question isolated from data-loading concerns.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from cloelib.cosmology.jax_cosmology import JAXBackground, JAXNonLinearPerturbations
from cloelib.observables.photo import ShearTracer
from cloelib.observables.photo.shear import PBJTATTLoopComputer
from cloelib.summary_statistics.angular_two_point import AngularTwoPoint

jax.config.update("jax_enable_x64", True)  # cloelib assumes float64 throughout

print("JAX backend:", jax.default_backend())

JAX backend: cpu


In [2]:
# Tracer redshift grid, a synthetic single-bin Gaussian n(z), and the
# (z, k) tabulation grid the Limber integral interpolates against.
z_tracer = jnp.linspace(0.2, 2.0, 15)
n_z_bins = 1
dndz = jnp.exp(-((z_tracer - 0.9) / 0.35) ** 2)[None, :]
dndz = dndz / jnp.trapezoid(dndz, z_tracer, axis=1)[:, None]

z_grid = jnp.linspace(0.01, 3.0, 100)
ks = jnp.logspace(-4, 1, 100)
ells = jnp.logspace(1.0, jnp.log10(200), 6)

nuisance_shear = {
    **{f"multiplicative_bias_{i + 1}": 0.0 for i in range(n_z_bins)},
    **{f"dz_shear_{i + 1}": 0.0 for i in range(n_z_bins)},
    **{f"width_shear_{i + 1}": 1.0 for i in range(n_z_bins)},
    "AIA": 1.0, "CIA": 0.0134, "EtaIA": -0.41,
}


def build_perturbations(H0=67.7, Omega_cdm0=0.25, Omega_b0=0.05, As=2e-9, ns=0.96):
    background = JAXBackground(
        H0=H0, Omega_b0=Omega_b0, Omega_cdm0=Omega_cdm0, Omega_k0=0.0,
        As=As, ns=ns, mnu=0.06, w0=-1.0, wa=0.0, gamma_MG=0.0, N_mnu=1,
    )
    perturbations = JAXNonLinearPerturbations(background=background)
    # JAXNonLinearPerturbations evaluates P(k,z) on the fly - it has no
    # natural pre-tabulated grid the way CAMB/HMcode2020Emu do, so `.z`/`.k`
    # (the informal-but-load-bearing contract AngularTwoPoint.get_Cl relies
    # on) have to be set explicitly, mirroring what those backends set
    # internally.
    perturbations.z = z_grid
    perturbations.k = ks
    return perturbations


perturbations = build_perturbations()
print("perturbations.linearperturbations:", type(perturbations.linearperturbations).__name__)

perturbations.linearperturbations: JAXLinearPerturbations


### Landmine #1 (found and fixed): `JAXBackground.Omega_m(0.0)`

`ShearTracer.get_window_lensing`/`get_window_IA`/`get_window_magnification` all call `self.background.Omega_m(0.0)` - a bare scalar. `JAXBackground.Omega_m` used to be implemented as a Python list comprehension, `[... for z in zs]`, which raises `TypeError: 'float' object is not iterable` on a bare float. Every other backend's `Omega_m` already tolerated this. Fixed in `cloelib/cosmology/jax_cosmology.py` by rewriting it as plain broadcastable `jnp` arithmetic (also removes an unnecessary Python-level loop). Confirmed below that `ShearTracer` now builds and runs against the pure-JAX backend.

In [3]:
tracer_she = ShearTracer(
    perturbations=perturbations, dndz=dndz, z=z_tracer, nuisance_params=nuisance_shear,
    ia_model="NLA",
)
window = tracer_she.get_window(z_tracer)
print("get_window shape:", window.shape, " finite:", bool(jnp.all(jnp.isfinite(window))))


get_window shape: (1, 15)  finite: True


## Does `get_Cl()` itself run, and is it differentiable?

`get_Cl()` runs. Whether it's *differentiable* is a separate question - let's ask `jax.grad` directly, differentiating a scalar built from the full public call.

In [4]:
two_point = AngularTwoPoint(tracer_she, tracer_she)
cells_nla = two_point.get_Cl(ells, 0, ks)
ee_nla = np.asarray(cells_nla[("SHE", "SHE", 1, 1)].array[0, 0])
print("get_Cl runs: shape", ee_nla.shape, " finite:", np.all(np.isfinite(ee_nla)))

get_Cl runs: shape (6,)  finite: True


In [5]:
def loss_via_public_api(AIA):
    perturbations = build_perturbations()
    nuisance = {**nuisance_shear, "AIA": AIA}
    tracer = ShearTracer(
        perturbations=perturbations, dndz=dndz, z=z_tracer, nuisance_params=nuisance,
        ia_model="NLA",
    )
    cl = AngularTwoPoint(tracer, tracer).get_Cl(ells, 0, ks)
    return jnp.sum(cl[("SHE", "SHE", 1, 1)].array)


try:
    grad_AIA_public = jax.grad(loss_via_public_api)(1.0)
    print(f"get_Cl() is directly differentiable: d(sum Cl)/d(AIA) = {grad_AIA_public:.6e}")
except Exception as e:
    print(f"get_Cl() is NOT directly differentiable: {type(e).__name__}")
    print(str(e).splitlines()[0])


get_Cl() is directly differentiable: d(sum Cl)/d(AIA) = -1.991709e-09


### Landmine #2 (found, then fixed upstream in `cosmolib`): the packaging step

`get_Cl` ends by wrapping `C_ell_calc` into a `dict` of `cosmolib.data.photo.AngularPowerSpectrum` objects (`_package_cl`, `angular_two_point.py`). That dataclass's `__post_init__` used to do `np.asarray(self.array, dtype=float)` unconditionally - a **plain NumPy** cast, in `cosmolib` (an external, pre-existing dependency, not cloelib's own code). Calling `np.asarray` on a value being traced by `jax.grad` raises `TracerArrayConversionError`: JAX cannot see through a step that forces its traced array into concrete NumPy.

This was a real boundary, not a subtle cloelib bug - `cosmolib`'s output dataclass was never designed with `jax.grad` in mind. It's now fixed directly in `cosmolib` (branch `26-fix-jax-clash-with-cloelib-photo-classes`): `__post_init__` branches on `isinstance(array, jax.Array)` (true for both concrete JAX arrays and tracers) and uses `jax.numpy.asarray` on that path instead, leaving the plain-NumPy path completely unchanged for every existing caller. `cloelib`'s `AngularTwoPoint` also grew a public `get_Cl_tensor(ells, nl, ks)` method around the same time, returning the same computation as `get_Cl` without the packaging step - it's no longer needed as a differentiability workaround, but it's still the leaner choice (skips building the `dict`/`AngularPowerSpectrum` objects) for anything running under `jax.jit`/`jax.vmap`. The cells below use `get_Cl_tensor` for exactly that reason, not because `get_Cl` doesn't work.


In [6]:
def raw_cl_sum(AIA=1.0, H0=67.7, Omega_cdm0=0.25):
    perturbations = build_perturbations(H0=H0, Omega_cdm0=Omega_cdm0)
    nuisance = {**nuisance_shear, "AIA": AIA}
    tracer = ShearTracer(
        perturbations=perturbations, dndz=dndz, z=z_tracer, nuisance_params=nuisance,
        ia_model="NLA",
    )
    two_point = AngularTwoPoint(tracer, tracer)
    C_ell_calc = two_point.get_Cl_tensor(ells, 0, ks)
    return jnp.sum(C_ell_calc)


print("finite check:", bool(jnp.isfinite(raw_cl_sum())))

val, grad_AIA = jax.value_and_grad(raw_cl_sum, argnums=0)(1.0)
print(f"d(sum Cl)/d(AIA) = {grad_AIA:.6e}   (nuisance parameter, NLA)")

val, grad_H0 = jax.value_and_grad(raw_cl_sum, argnums=1)(1.0, 67.7)
print(f"d(sum Cl)/d(H0)  = {grad_H0:.6e}   (cosmological parameter, through JAXBackground + JAXNonLinearPerturbations)")

val, grad_Ocdm = jax.value_and_grad(raw_cl_sum, argnums=2)(1.0, 67.7, 0.25)
print(f"d(sum Cl)/d(Omega_cdm0) = {grad_Ocdm:.6e}")


finite check: True
d(sum Cl)/d(AIA) = -1.991709e-09   (nuisance parameter, NLA)
d(sum Cl)/d(H0)  = 9.121384e-10   (cosmological parameter, through JAXBackground + JAXNonLinearPerturbations)
d(sum Cl)/d(Omega_cdm0) = 2.585775e-07


Both gradients are finite, real numbers - the Limber integral, the growth-factor ODE solve, halofit, the window functions, all differentiate cleanly through `jax.grad`. NLA's full physics chain is autodifferentiable with the pure-JAX backend, all the way from $C_\ell$ back to $H_0$.

A quick finite-difference cross-check, since a finite gradient isn't proof it's the *correct* gradient:

In [7]:
eps = 1e-4
fd_H0 = (raw_cl_sum(1.0, 67.7 + eps) - raw_cl_sum(1.0, 67.7 - eps)) / (2 * eps)
print(f"jax.grad   d(sum Cl)/d(H0) = {grad_H0:.6e}")
print(f"finite-diff d(sum Cl)/d(H0) = {fd_H0:.6e}")
print(f"relative difference: {abs((grad_H0 - fd_H0) / fd_H0):.2e}")

jax.grad   d(sum Cl)/d(H0) = 9.121384e-10
finite-diff d(sum Cl)/d(H0) = 9.121384e-10
relative difference: 1.63e-10


## TATT: differentiable in its amplitudes, not through its kernels

`TATTContribution`'s amplitude functions ($C_1(z)$, $C_{1\delta}(z)$, $C_2(z)$, `compute_kernel`) are all plain `jnp` math - same story as NLA. But its ten one-loop kernels come from `PBJTATTLoopComputer`, which calls `fastpt.FASTPT.IA_ta`/`.IA_tt`/`.IA_mix` - **NumPy/SciPy, not JAX** (FAST-PT does its own FFTLog internally; it has no JAX implementation). Two different predictions to check:

- Differentiating with respect to a **TATT amplitude parameter** (`A2IA`, `bTA`, ...) should work: the kernel *values* are fixed multiplicative weights as far as these parameters are concerned, computed once and never touched again.
- Differentiating with respect to a **cosmological parameter** that the kernels themselves depend on (through the linear $P(k)$ FAST-PT consumes) should *not* work: gradient would have to flow back through FAST-PT's plain-NumPy internals, which JAX cannot trace.

In [8]:
nuisance_shear_tatt = {**nuisance_shear, "A2IA": 0.40, "bTA": -0.83, "Eta2IA": 2.69, "z0IA": 0.62}


def build_tatt_tracer(perturbations, A2IA=0.40, bTA=-0.83):
    nuisance = {**nuisance_shear_tatt, "A2IA": A2IA, "bTA": bTA}
    return ShearTracer(
        perturbations=perturbations, dndz=dndz, z=z_tracer, nuisance_params=nuisance,
        ia_model="TATT", tatt_loop_computer=PBJTATTLoopComputer(perturbations),
    )


tracer_she_tatt = build_tatt_tracer(perturbations)
two_point_tatt = AngularTwoPoint(tracer_she_tatt, tracer_she_tatt)
C_ell_tatt = two_point_tatt.get_Cl_tensor(ells, 0, ks)
print("TATT raw Cl finite:", bool(jnp.all(jnp.isfinite(C_ell_tatt))), " sum:", float(jnp.sum(C_ell_tatt)))


TATT raw Cl finite: True  sum: 5.6564819799684086e-08


In [9]:
def raw_cl_sum_tatt_amplitude(A2IA):
    tracer = build_tatt_tracer(perturbations, A2IA=A2IA)
    two_point = AngularTwoPoint(tracer, tracer)
    return jnp.sum(two_point.get_Cl_tensor(ells, 0, ks))


val, grad_A2IA = jax.value_and_grad(raw_cl_sum_tatt_amplitude)(0.40)
print(f"d(sum Cl_TATT)/d(A2IA) = {grad_A2IA:.6e}   -> finite: {bool(jnp.isfinite(grad_A2IA))}")

eps = 1e-4
fd_A2IA = (raw_cl_sum_tatt_amplitude(0.40 + eps) - raw_cl_sum_tatt_amplitude(0.40 - eps)) / (2 * eps)
print(f"finite-diff             = {fd_A2IA:.6e}")
print(f"relative difference: {abs((grad_A2IA - fd_A2IA) / fd_A2IA):.2e}")


d(sum Cl_TATT)/d(A2IA) = 4.352458e-11   -> finite: True
finite-diff             = 4.352458e-11
relative difference: 3.29e-11


In [10]:
def raw_cl_sum_tatt_cosmology(H0):
    perturbations_h = build_perturbations(H0=H0)
    tracer = build_tatt_tracer(perturbations_h)
    two_point = AngularTwoPoint(tracer, tracer)
    return jnp.sum(two_point.get_Cl_tensor(ells, 0, ks))


try:
    jax.grad(raw_cl_sum_tatt_cosmology)(67.7)
    print("Unexpectedly differentiable through the FAST-PT kernels.")
except Exception as e:
    print(f"NOT differentiable through TATT's one-loop kernels: {type(e).__name__}")
    print(str(e).splitlines()[0])
    print("\nWhy: PBJTATTLoopComputer._kernels_for calls "
          "linear_perturbations.matter_power_spectrum(...) (traced, fine) "
          "and then np.asarray(...) on the result to hand it to FAST-PT "
          "(fastpt.FASTPT.IA_ta/.IA_tt/.IA_mix, plain NumPy/SciPy) - that "
          "cast is exactly where JAX's trace is severed.")


NOT differentiable through TATT's one-loop kernels: TracerArrayConversionError
The numpy.ndarray conversion method __array__() was called on traced array with shape float64[100]

Why: PBJTATTLoopComputer._kernels_for calls linear_perturbations.matter_power_spectrum(...) (traced, fine) and then np.asarray(...) on the result to hand it to FAST-PT (fastpt.FASTPT.IA_ta/.IA_tt/.IA_mix, plain NumPy/SciPy) - that cast is exactly where JAX's trace is severed.


## Summary

| | differentiable via `jax.grad`? |
|---|---|
| `get_Cl()` (public API, NLA) | **Yes** - fixed upstream in `cosmolib` (see Landmine #2) |
| `get_Cl_tensor()` (pre-packaging tensor, NLA or TATT) | **Yes** |
| NLA nuisance parameters (`AIA`, `CIA`, `EtaIA`) | **Yes** |
| NLA w.r.t. cosmological parameters (`H0`, `Omega_cdm0`, ...) via `JAXBackground`/`JAXNonLinearPerturbations` | **Yes** - confirmed against finite differences |
| TATT amplitude parameters (`A2IA`, `bTA`, ...) | **Yes** - confirmed against finite differences |
| TATT w.r.t. cosmological parameters that affect its one-loop kernels | **No** - FAST-PT (`PBJTATTLoopComputer`) is plain NumPy/SciPy, not JAX |

Three real gaps surfaced by actually running this, not by inspection alone:

1. **Fixed** (`cloelib`): `JAXBackground.Omega_m` couldn't accept the bare scalar `ShearTracer` calls it with - a real bug independent of autodiff, now fixed (vectorized `jnp` arithmetic, no behavior change for the array inputs it already handled).
2. **Fixed** (`cosmolib`, branch `26-fix-jax-clash-with-cloelib-photo-classes`): `get_Cl()`'s own packaging step depended on an external dataclass (`cosmolib.data.photo.AngularPowerSpectrum`) that wasn't JAX-aware. `__post_init__` now uses `jax.numpy.asarray` when given a JAX array/tracer instead of unconditionally forcing NumPy, so the public `get_Cl()` differentiates directly - no workaround needed any more. `cloelib` also gained a public `get_Cl_tensor()` out of this investigation, which remains useful on its own merits (skips packaging overhead) even though it's no longer required for gradients.
3. **Structural, not a bug**: TATT's use of a real (non-JAX) perturbation-theory backend for its one-loop kernels is an inherent limit - FAST-PT's FFTLog implementation has no JAX port to fall back on. A gradient through TATT's *own physical model* is real and correct wherever it's asked for (any amplitude parameter); a gradient through *cosmology-dependent kernel shapes* would need a JAX-native one-loop PT implementation, which doesn't exist yet for `cloelib`.
